# blockperm: a two-minute demo

When conditions come in blocks of consecutive trials and the cross-validation puts trials of the same block in both the training and the test set, slow drift can make the blocks, and so the labels, look decodable. This notebook assigns labels with no information to whole blocks of a synthetic drifting recording and compares scikit-learn's `permutation_test_score` (trial labels permuted) with `block_permutation_test_score` (block labels permuted, whole cross-validation rerun). A valid test should reject in about 5% of recordings.

Runs on the free Colab CPU in about two minutes. Documentation: https://github.com/jinchoi7651507/block-permutation-decoding/tree/main/blockperm

In [ ]:
%pip install -q blockperm

## A synthetic recording with slow drift

Poisson counts from 20 units share one slow factor. Twelve blocks of 20 trials each; half of the blocks are labelled 1 at random, so the labels carry no information about the activity.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, permutation_test_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from blockperm import block_permutation_test_score


def recording(rng, n_blocks=12, block_len=20, n_units=20, tau=20.0, sd=0.6):
    """Poisson counts with one slow shared factor, and labels constant within blocks (half the blocks each)."""
    n = n_blocks * block_len
    a = np.exp(-1.0 / tau)
    z = np.empty(n)
    z[0] = rng.standard_normal()
    e = rng.standard_normal(n) * np.sqrt(1 - a * a)
    for t in range(1, n):
        z[t] = a * z[t - 1] + e[t]
    X = rng.poisson(np.exp(np.log(5.0) + sd * z[:, None] * rng.standard_normal(n_units)[None, :])).astype(float)
    blocks = np.repeat(np.arange(n_blocks), block_len)
    lab = np.zeros(n_blocks, int)
    lab[rng.choice(n_blocks, n_blocks // 2, replace=False)] = 1
    return X, lab[blocks], blocks


est = Pipeline([("scale", StandardScaler()), ("lr", LogisticRegression(max_iter=1000))])

## One recording

`KFold(5, shuffle=True)` puts trials of every block in both training and test folds. The only change between the two tests is the function name and `blocks=`.

In [ ]:
rng = np.random.default_rng(0)
X, y, blocks = recording(rng)
cv = KFold(5, shuffle=True, random_state=0)

score, _, p_trial = permutation_test_score(est, X, y, cv=cv, n_permutations=99, random_state=0)
_, _, p_block = block_permutation_test_score(est, X, y, blocks=blocks, cv=cv, n_permutations=99, random_state=0)
print("cross-validated accuracy: %.2f" % score)
print("p, trial labels permuted (permutation_test_score):  %.3f" % p_trial)
print("p, block labels permuted (block_permutation_test_score): %.3f" % p_block)

## Forty recordings: how often does each test reject?

This is the example script `examples/fake_block_labels.py` of the repository. With the versions listed in the README it prints 14 (35%) for the trial-level test and 1 (2%) for the block permutation test.

In [ ]:
n_rec, n_perm = 40, 99
trial = block = 0
for r in range(n_rec):
    rng = np.random.default_rng(r)
    X, y, blocks = recording(rng)
    cv = KFold(5, shuffle=True, random_state=r)
    _, _, p_t = permutation_test_score(est, X, y, cv=cv, n_permutations=n_perm, random_state=r)
    _, _, p_b = block_permutation_test_score(est, X, y, blocks=blocks, cv=cv, n_permutations=n_perm, random_state=r)
    trial += p_t < 0.05
    block += p_b < 0.05
print("labels with no information, %d recordings, p < 0.05:" % n_rec)
print("  permutation_test_score (trial labels permuted): %d (%.0f%%)" % (trial, 100 * trial / n_rec))
print("  block_permutation_test_score (block labels permuted): %d (%.0f%%)" % (block, 100 * block / n_rec))

## On your own data

```python
from blockperm import block_permutation_test_score
score, null_scores, p = block_permutation_test_score(
    pipeline, X, y, blocks=block_id, cv=cv, n_permutations=1000, random_state=0)
```

- `y` must be constant within each block. Put every step fitted to the labels (scaling, feature or neuron selection, the model) in one `Pipeline`, because the whole cross-validation is rerun in each permutation.
- The test is exact when conditions were assigned to blocks at random. With a fixed block order it assumes exchangeable blocks.
- `strata=` permutes within runs or sessions, `draw=` takes your own generator of block labels (for example pseudosessions), and with few blocks every arrangement is enumerated.
- More in the README and `docs/TUTORIAL.md` of the repository.